In [19]:
import pandas as pd
import numpy as np
import json, os, math
import joblib
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from NeuralSelector.index import NeuralSelector

In [20]:
OPTIONS = json.loads(open('info.json', 'r').read())
dt      = OPTIONS.get('dt')

x_min, x_max = OPTIONS.get('sensor_range')
u_min, u_max = OPTIONS.get('actuator_range')
OFFSET = OPTIONS.get('offset')
ts     = OPTIONS.get('ts')
u_std  = OPTIONS.get('actuator_std')
max_action = OPTIONS.get('max_action')

- Loading System Simulator

In [21]:
system_info  = json.loads(open('Saved/System/info.json', 'r').read())
system_vars  = system_info['variables']
n_states_sys = sum(['x(n-' in var for var in system_vars]) + 1
n_states_ctl = sum(['u(n-' in var for var in system_vars]) + 1

print(system_info)
print(system_vars)
print(n_states_sys, 'system past states')
print(n_states_ctl, 'actuator past states')

system = joblib.load('Saved/System/model.pkl')
system

{'model': 'linear_regression', 'params': {'memory': 'None', 'steps': "[('scaler', StandardScaler()), ('model', LinearRegression())]", 'transform_input': 'None', 'verbose': 'False', 'scaler': 'StandardScaler()', 'model': 'LinearRegression()', 'scaler__copy': 'True', 'scaler__with_mean': 'True', 'scaler__with_std': 'True', 'model__copy_X': 'True', 'model__fit_intercept': 'True', 'model__n_jobs': 'None', 'model__positive': 'False', 'model__tol': '1e-06'}, 'K_CV': 5, 'info': {'r2': 0.9962738131, 'r2_adj': 0.9961557689, 'rmse': 0.0043166528, 'mae': 0.0016836612}, 'variables': ['u', 'u(n-1)', 'u(n-2)', 'u(n-3)', 'u(n-4)', 'u(n-5)', 'u(n-6)', 'u(n-7)', 'u(n-8)', 'u(n-9)', 'u(n-10)', 'u(n-11)', 'u(n-12)', 'u(n-13)', 'u(n-14)', 'u(n-15)', 'u(n-16)', 'u(n-17)', 'u(n-18)', 'u(n-19)', 'u(n-20)', 'u(n-21)', 'u(n-22)', 'u(n-23)', 'u(n-24)', 'u(n-25)', 'u(n-26)', 'x(n-1)', 'x(n-2)', 'x(n-3)', 'x(n-4)', 'x(n-5)', 'x(n-6)', 'x(n-7)', 'x(n-8)', 'x(n-9)', 'x(n-10)', 'x(n-11)', 'x(n-12)', 'x(n-13)', 'x(n-

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


## SIMULADOR EM MALHA FECHADA
`StatesUpdater` armazena valores defasados $y(k-1), u(k-1)$ necessários para lidar com inércia e atrasos de transporte.

In [22]:
class StatesUpdater:
    def __init__(self, size, initial=0.00):
        self.initial = initial
        self.size = size
        self.buffer = np.array([initial for _ in range(size)])
        
    def update(self, value):
        for i in range(self.size-1, 0, -1):
            self.buffer[i] = self.buffer[i-1]
        self.buffer[0] = value
        return self.buffer
    
    def get(self, var='x'):
        data = {var: self.buffer[0]}
        for i in range(1, self.size):
            data[f'{var}(n-{i})'] = self.buffer[i]
        return {key: float(val) for key, val in data.items()}


states = StatesUpdater(5)
for i in range(5): print(states.update(i))

[0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0.]
[2. 1. 0. 0. 0.]
[3. 2. 1. 0. 0.]
[4. 3. 2. 1. 0.]


**Simulador de Malha Fechada:** Simula o sistema de malha fechada. Retorna o estado atual, a recompensa (baseada em precisão e suavidade) 
    e se o episódio foi finalizado (se a bola caiu da barra).

Exemplo: Se a sua barra tem 0.3m, o erro máximo é 0.3m. $1 / (0.3^2) = 11.1$.Se o motor pode variar no máximo 2 graus por passo de controle. $1 / (2^2) = 0.25$. Chute inicial ótimo: errorWeight = 11.1 e effortWeight = 0.25.

In [23]:
class FeedbackSimulator:
    def __init__(self, system_model, system_vars, n_states_sys=5, n_states_ctl=5, w_err=50.0, w_eff=0.1, max_steps=200, max_action=5):
        self.system_model = system_model
        self.system_vars  = system_vars
        self.max_action = max_action
        
        self.n_states_sys = n_states_sys
        self.n_states_ctl = n_states_ctl
        
        self.w_err = w_err
        self.w_eff = w_eff
        self.max_steps = max_steps

        self.integral_err  = 0.0
        self.integral_clip = ((x_max - x_min) / 2.0) * ts
        self.prev_delta_u  = 0.0
        
        # 1. Definindo o valor inicial dinamicamente no centro das faixas
        sensor_initial   = (x_max + x_min) / 2.0
        actuator_initial = (u_max + u_min) / 2.0
        
        self.sensor   = StatesUpdater(self.n_states_sys, initial=sensor_initial)
        self.actuator = StatesUpdater(self.n_states_ctl, initial=actuator_initial)
        
        # 2. Setpoint inicial aleatório dentro da capacidade do sensor
        self.setpoint = np.random.uniform(x_min, x_max)
        self.current_step = 0

    def reset(self):
        self.current_step = 0
        sensor_initial    = (x_max + x_min) / 2.0
        actuator_initial  = (u_max + u_min) / 2.0

        self.integral_err = 0.0
        self.integral_clip = ((x_max - x_min) / 2.0) * ts
        self.prev_delta_u = 0.0

        self.sensor   = StatesUpdater(self.n_states_sys, initial=sensor_initial)
        self.actuator = StatesUpdater(self.n_states_ctl, initial=actuator_initial)
        
        self.setpoint = np.random.uniform(x_min, x_max) # Novo alvo aleatório possível de ser atingido
        return self.getStates()
    
    def getStates(self):
        sensor_center    = (x_max + x_min) / 2.0
        sensor_amplitude = (x_max - x_min) / 2.0
        
        actuator_center    = (u_max + u_min) / 2.0
        actuator_amplitude = (u_max - u_min) / 2.0
        
        error = self.setpoint - self.sensor.buffer[0]
        normalized_error = error / sensor_amplitude
        
        integral_norm = np.clip(self.integral_err / self.integral_clip, -1.0, 1.0)
        state = [normalized_error, integral_norm]   # ERA: state = [normalized_error]
        
        for i in range(self.n_states_sys): # Histórico do sensor
            normalized_x = (self.sensor.buffer[i] - sensor_center) / sensor_amplitude
            state.append(normalized_x)
            
        for i in range(self.n_states_ctl): # Histórico do atuador
            normalized_u = (self.actuator.buffer[i] - actuator_center) / actuator_amplitude
            state.append(normalized_u)
            
        return np.array(state, dtype=np.float32)
    
    def step(self, action):
        delta_u = action[0]
        new_u   = self.actuator.buffer[0] + delta_u
        
        if new_u < u_min: 
            new_u = u_min
        
        if new_u > u_max: 
            new_u = u_max

        self.actuator.update(new_u)
        data = self.actuator.get('u')
        
        for i in range(1, self.n_states_sys):
            data[f'x(n-{i})'] = self.sensor.buffer[i-1]
        
        model_input = pd.DataFrame([data], columns=self.system_vars)
        next_x      = self.system_model.predict(model_input)[0]
        
        self.sensor.update(next_x)
        self.current_step += 1

        if self.current_step % int(ts / dt) == 0:
            self.setpoint = np.random.uniform(x_min, x_max)
        
        error = self.setpoint - next_x
        jerk  = delta_u - self.prev_delta_u
        self.prev_delta_u = delta_u

        reward = -1 * (self.w_err * (error**2) + self.w_eff * np.abs(delta_u) + 0.10 * (jerk / (2 * self.max_action))**2)
        states = self.getStates()
        done   = False
        
        if next_x < x_min or next_x > x_max: # Punição = pior erro possível ao quadrado vezes o total de passos restantes
            reward -= (self.w_err * 2) * (self.max_steps - self.current_step)
            done = True
        
        if self.current_step >= self.max_steps:
            done = True

        self.integral_err = float(np.clip(
            self.integral_err + error * dt,
            -self.integral_clip, self.integral_clip
        ))
                    
        return states, reward, done


feedback_options = {
    'system_model': system,
    'system_vars':  system_vars,
    'n_states_sys': n_states_sys,
    'n_states_ctl': n_states_ctl,
    'w_err': 1/(x_max**2),
    'w_eff': 1/(max_action**2),
    'max_steps': int(4 * ts/dt),
    'max_action': max_action
}

feedback = FeedbackSimulator(**feedback_options)
state = feedback.reset()
print(feedback_options)
state

{'system_model': Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())]), 'system_vars': ['u', 'u(n-1)', 'u(n-2)', 'u(n-3)', 'u(n-4)', 'u(n-5)', 'u(n-6)', 'u(n-7)', 'u(n-8)', 'u(n-9)', 'u(n-10)', 'u(n-11)', 'u(n-12)', 'u(n-13)', 'u(n-14)', 'u(n-15)', 'u(n-16)', 'u(n-17)', 'u(n-18)', 'u(n-19)', 'u(n-20)', 'u(n-21)', 'u(n-22)', 'u(n-23)', 'u(n-24)', 'u(n-25)', 'u(n-26)', 'x(n-1)', 'x(n-2)', 'x(n-3)', 'x(n-4)', 'x(n-5)', 'x(n-6)', 'x(n-7)', 'x(n-8)', 'x(n-9)', 'x(n-10)', 'x(n-11)', 'x(n-12)', 'x(n-13)', 'x(n-14)', 'x(n-15)', 'x(n-16)', 'x(n-17)', 'x(n-18)', 'x(n-19)', 'x(n-20)', 'x(n-21)', 'x(n-22)', 'x(n-23)', 'x(n-24)', 'x(n-25)', 'x(n-26)'], 'n_states_sys': 27, 'n_states_ctl': 27, 'w_err': 11.11111111111111, 'w_eff': 0.01, 'max_steps': 643, 'max_action': 10.0}


array([0.6325004, 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       ], dtype=float32)

## AGENTE DRL
Essas classes encapsulam a essência algorítmica do DDPG (Off-Policy). Elas agem baseadas nos tensores injetados pelo `NeuralSelector` e mantêm o buffer de replay das experiências passadas, vital para descorrelacionar os dados de treinamento.

In [24]:
class SystemAgent:
    def __init__(self, actor, evaluator, state_dim, action_dim, max_action, 
                 actor_kwargs={}, critic_kwargs={}, actor_lr=1e-4, critic_lr=1e-3, memory_size=1e5):
        
        # --- Redes Neurais e Otimizadores ---
        self.actor = actor(state_dim, action_dim, max_action, **actor_kwargs)
        self.actor_target = actor(state_dim, action_dim, max_action, **actor_kwargs)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=actor_lr)

        self.critic = evaluator(state_dim, action_dim, **critic_kwargs)
        self.critic_target = evaluator(state_dim, action_dim, **critic_kwargs)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.max_action = max_action
        self.discount   = math.exp(-dt / ts)
        self.tau        = 0.005
        self.storage, self.max_mem, self.ptr = [], int(memory_size), 0

    def store(self, *transition):
        """Empacota state, action, next_state, reward, done dinamicamente."""
        if len(self.storage) < self.max_mem:
            self.storage.append(transition)
        else:
            self.storage[self.ptr] = transition
        self.ptr = (self.ptr + 1) % self.max_mem

    def _sample_memory(self, batch_size):
        """Usa zip(*...) para desempacotar a matriz inteira em 1 linha."""
        idx = np.random.randint(0, len(self.storage), size=batch_size)
        batch = [self.storage[i] for i in idx]
        
        # Transpõe as colunas e converte tudo direto para Tensor
        s, a, s_, r, d = map(lambda x: torch.FloatTensor(np.array(x)), zip(*batch))
        return s, a, s_, r.unsqueeze(1), d.unsqueeze(1)

    def select_action(self, state):
        with torch.no_grad():
            return self.actor(torch.FloatTensor(state).reshape(1, -1)).cpu().numpy().flatten()

    def train(self, batch_size=64):
        s, a, s_, r, d = self._sample_memory(batch_size)

        # --- Treino do Critic ---
        target_q = r + (1 - d) * self.discount * self.critic_target(s_, self.actor_target(s_))
        critic_loss = F.mse_loss(self.critic(s, a), target_q.detach())
        
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # --- Treino do Actor ---
        actor_loss = -self.critic(s, self.actor(s)).mean()
        
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        # --- Soft Update Agrupado ---
        for net, target_net in [(self.critic, self.critic_target), (self.actor, self.actor_target)]:
            for p, tp in zip(net.parameters(), target_net.parameters()):
                tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

## TREINAMENTO
A classe trainer orquestra a interação entre o Agente e o Simulador da Planta. Executa os episódios e preenche a Memória de Experiência do Agente, exportando os resultados.

In [25]:
class Trainer:
    def __init__(self, agent, env, batch_size=64, epochs=100):
        self.agent = agent
        self.feedback   = env
        self.max_action = agent.max_action
        self.noise_std  = agent.max_action * 0.3
        self.batch_size = batch_size
        self.action_dim = 1
        self.epochs = epochs

    def start(self):
        for ep in tqdm(range(self.epochs)):
            state = self.feedback.reset()
            
            for t in range(self.feedback.max_steps): 
                sigma  = self.max_action * max(0.03, self.noise_std * (1 - ep / self.epochs))
                noise  = np.random.normal(0, sigma, size=self.action_dim)
                action = (self.agent.select_action(state) + noise).clip(-self.max_action, self.max_action)
                
                next_state, reward, done = self.feedback.step(action)
                self.agent.store(state, action, next_state, reward, done)
                state = next_state
                
                if len(self.agent.storage) > self.batch_size:
                    self.agent.train(self.batch_size)
                    
                if done:
                    break

In [26]:
selector = NeuralSelector('lstm')
Actor, Evaluator, grid_params = selector.get()

batch_size = grid_params['batchSize'][0] 
print(f"Batch Size: {batch_size}")

networkKwargs = {key: values[0] for key, values in grid_params.items() if key not in ['actorLr', 'criticLr', 'batchSize']}

agent_options = {
    'actor':  Actor, 
    'evaluator': Evaluator, 
    'state_dim':  len(feedback.getStates()), 
    'action_dim': 1, 
    'max_action': max_action,
    'actor_kwargs':  networkKwargs,
    'critic_kwargs': networkKwargs,
    'actor_lr' : grid_params['actorLr'][0],
    'critic_lr': grid_params['criticLr'][0]
}

agent   = SystemAgent(**agent_options)
trainer = Trainer(agent, feedback, batch_size=batch_size, epochs=300) 
trainer.start()

Batch Size: 64


100%|██████████| 300/300 [20:51<00:00,  4.17s/it] 


In [27]:
os.makedirs('Saved/Controller/', exist_ok=True)
torch.save(agent.actor.state_dict(), 'Saved/Controller/actor.pth')

info = {
    'architecture': selector.chosen,
    'actor_kwargs': networkKwargs,
    'max_action': agent.max_action,
    'feedback_options': str(feedback_options),
    'agent_settings': str(agent_options)
}

with open('Saved/Controller/info.json', 'w') as f:
    json.dump(info, f, indent=2)

print("Weights and topology metadata saved!")

Weights and topology metadata saved!
